# Data Wrangling using Pandas

## Dataset Description: Student Assessment Questionnaires

The dataset `assessment_generated.csv` contains information derived from student assessment questionnaires.

Each record represents an individual student's response and includes demographic, academic, and self-assessment information. The dataset comprises the following attributes:

- **`timestamp`**  
  The date and time when the assessment was submitted, formatted as `yyyy-mm-dd hh:mm:ss timezone`.

- **`netid`**  
  The encoded NetID of the student. Valid NetIDs must have a string length between 8 and 14 characters (inclusive). Entries falling outside this range are considered invalid.

- **`ruid`**  
  The encoded RUID of the student. A valid RUID is expected to contain exactly 18 characters. Any deviation from this length is considered invalid.

- **`section`**  
  The course section number as reported by the student. This field may contain inaccuracies, as some students provided incorrect section information.

- **`role`**  
  The academic standing of the student. Possible values include:
  - `Freshman`
  - `Sophomore`
  - `Junior`
  - `Senior`
  - `Graduate`
  - `Other`

- **`major`**  
  The declared major of the student. Accepted categories are:
  - `Computer Science`
  - `Electrical and Computer Engineering`
  - `Mathematics`
  - `Other`

- **Skill Proficiency Columns**  
  The following columns record students’ self-assessed proficiency levels in specific skills, rated on scales ranging from 0 up to a multiple of 5 (depending on the number of questions per topic). Missing values are present in some entries.

  - `data_structures`  
  - `calculus_and_linear_algebra`  
  - `probability_and_statistics`  
  - `data_visualization`  
  - `python_libraries`  
  - `shell_scripting`  
  - `sql`  
  - `python_scripting`  
  - `jupyter_notebook`  
  - `regression`  
  - `programming_languages`  
  - `algorithms`  
  - `complexity_measures`  
  - `visualization_tools`  
  - `massive_data_processing`


## Tasks

- Import Data
  - Load assessment_generated.xlsx as a DataFrame named student_assessment_xlsx.
- Verify NetIDs
  - Ensure that student_assessment_xlsx contains the same set of netids as student_assessment.
- Analyze RUID Lengths
  - Display the frequency of each ruid length.
  - Display records where ruid length exceeds 20 characters.
- Compute Total Score
  - Create a new column total_score as the sum of all skill proficiency columns.
  - Sort records by total_score in descending order.
- Group Statistics by Section
  - Group students by section and compute the mean, median, and standard deviation for each skill proficiency column.
- Pivot Table by Role and Section
  - Create a pivot table where:
    - Rows correspond to role
    - Columns correspond to section numbers
    - Entries contain the average total_score
- Format Timestamp
  - Convert timestamp values to the EST timezone instead of UTC.
- Normalize Skill Proficiency
  - For each proficiency column, apply z-score normalization rather than min-max scaling.
- Handle Missing Values
  - Fill missing values in each skill proficiency column with the column mean.
- Remove Duplicate Records
  - Drop duplicates while keeping only the record with the highest total_score.
- Resolve Swapped IDs
  - Identify records where students may have swapped netid and ruid. (Hint: netid should be shorter than ruid.)
- For records swapped netid and ruid, attempt a join using:
  - student_assessment.ruid = student_list.netid


### Setup Code (Please run this first to set up the environment)

In [ ]:
import numpy as np
import pandas as pd
from sklearn import preprocessing

### Import Data

In [ ]:
def import_data(path):
    """
    Load assessment_generated.xlsx as a DataFrame 
    IN: path, str, path to the file
    OUT: student_assessment_xlsx, pd.DataFrame
    """
    # Your_Code_Here
    student_assessment_xlsx = pd.read_excel(path)
    # print(student_assessment_xlsx.head())

    return student_assessment_xlsx

### Verify NetIDs

In [ ]:
def verify_netids(df_csv, df_xlsx):
    """
    Verify that the NetIDs in the CSV and XLSX files match
    IN: df_csv, pd.DataFrame, dataframe from CSV file
        df_xlsx, pd.DataFrame, dataframe from XLSX file
    OUT: flag, bool, True if NetIDs match, False otherwise
    """
    # Your_Code_Here
    flag = set(df_csv['netid']) == set(df_xlsx['netid'])# netid column is all lowercase

    return flag

### Analyze RUID Lengths

In [ ]:
def get_ruid_length_freq(df):
    """
    Get the frequency of RUID lengths in the dataframe
    IN: df, pd.DataFrame, dataframe containing RUIDs
    OUT: ruid_length_freq, dict, dictionary with RUID lengths as keys and their frequencies as values
    """
    # Your_Code_Here
    lengths = df['ruid'].astype(str).str.len() # Transform ruid column to string and get length of each ruid
    ruid_length_freq = lengths.groupby(lengths).size().to_dict() # Group by length and count frequency then convert into dictionary
    return ruid_length_freq

In [ ]:
def get_ruid_length_outliers(df):
    """
    Identify RUIDs with lengths that are > 20 characters
    IN: df, pd.DataFrame, dataframe containing RUIDs
    OUT: outliers, pd.DataFrame, dataframe containing outlier RUIDs
    """
    # Your_Code_Here
    outliers = df[df['ruid'].astype(str).str.len() > 20] # Filter a DF containing RUIDs with length > 20
    # There was only 1 outlier from the previous result

    return outliers


### Compute Total Score

Note: Skip `nan` entries when computing sum.

In [ ]:
def compute_total_score(df, skill_columns):
    """
    Compute the total score for each student
    IN: df, pd.DataFrame, dataframe containing student assessments
    IN: skill_columns, list of str, list of columns to sum for total score
    OUT: df, pd.DataFrame, dataframe with total_score column
    """
    # Your_Code_Here
    df['total_score'] = df[skill_columns].sum(axis=1) # Axis = 0 goes down entries, Axis = 1 goes across features
    
    return df

In [ ]:
def sort_by_total_score(df):
    """
    Sort the dataframe by total score in descending order
    IN: df, pd.DataFrame, dataframe containing total_score column
    OUT: df_sorted, pd.DataFrame, sorted dataframe
    """
    # Your_Code_Here
    df_sorted = df.sort_values(by='total_score', ascending=False) # Order the dataframe by descending total score column

    return df_sorted

### Group Statistics by Section

In [ ]:
def get_section_statistics(df, skill_columns):
    """
    Group students by section and compute the mean, median, and standard deviation for each skill proficiency column
    IN: df, pd.DataFrame, dataframe containing student assessments
    IN: skill_columns, list of str, list of skill proficiency columns
    OUT: section_stats, pd.DataFrame, dataframe with section statistics
    """
    # Your_Code_Here
    section_stats = df.groupby('section')[skill_columns].agg(
        ['mean', 'median', 'std']
    )
    return section_stats

### Pivot Table by Role and Section

In [ ]:
def create_pivot_table(df):
    """
    Create a pivot table where rows correspond to role, columns correspond to section numbers, and entries contain the average total_score
    IN: df, pd.DataFrame, dataframe containing student assessments
    OUT: pivot_table, pd.DataFrame, pivot table
    """
    # set role as categorical with specified order
    # Your_Code_Here

    # generate pivot table
    # Your_Code_Here
    role_order = ['Freshman', 'Sophomore', 'Junior','Senior', 'Graduate', 'Other']
    df['role'] = pd.Categorical(df['role'],categories=role_order,ordered=True)
    pivot_table = df.pivot_table(index='role',columns='section',values='total_score',aggfunc='mean',observed=False
    )
    
    
    return pivot_table

### Format Timestamp

In [ ]:
def format_timestamp(df):
    """
    Convert timestamp values to the EST timezone instead of UTC
    IN: df, pd.DataFrame, dataframe containing student assessments
    OUT: df, pd.DataFrame, dataframe with formatted timestamp
    """
    # Your_Code_Here
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_convert('EST')
    return df

### Normalize Skill Proficiency

In [ ]:
def normalize_skills(df, skill_columns):
    """
    For each proficiency column, apply z-score normalization
    IN: df, pd.DataFrame, dataframe containing student assessments
    IN: skill_columns, list of str, list of skill proficiency columns
    OUT: df, pd.DataFrame, dataframe with normalized skills
    """
    
    # Your_Code_Here
    scaler = preprocessing.StandardScaler()
    df[skill_columns] = scaler.fit_transform(df[skill_columns])
    return df

### Handle Missing Values

In [ ]:
def handle_missing_values(df, skill_columns):
    """
    Fill missing values in each skill proficiency column with the column mean
    IN: df, pd.DataFrame, dataframe containing student assessments
    OUT: df, pd.DataFrame, dataframe with missing values handled
    """
    # Your_Code_Here
    df[skill_columns] = df[skill_columns].fillna(df[skill_columns].mean())
    return df

### Remove Duplicate Records

Note: Drop duplicates based on `ruid`. Keep the record with the highest `total_score`. If there is a tie, keep the one whose `timestamp` is latest.

In [ ]:
def remove_duplicates(df):
    """
    Drop duplicates while keeping only the record with the highest total_score
    IN: df, pd.DataFrame, dataframe containing student assessments
    OUT: df, pd.DataFrame, dataframe with duplicates removed
    """
    # sort by total_score descending, then by timestamp descending
    # Your_Code_Here
    df = df.sort_values(by=['total_score', 'timestamp'],ascending=[False, False])

    # drop duplicates
    # Your_Code_Here
    df = df.drop_duplicates(subset='ruid', keep='first')
    
    # sort back
    # Your_Code_Here
    df = df.sort_index()

    return df

### Resolve Swapped IDs

In [ ]:
def get_swapped_records(df):
    """
    Identify records where students may have swapped netid and ruid
    IN: df, pd.DataFrame, dataframe containing student assessments
    OUT: swapped_df, pd.DataFrame, dataframe containing swapped records
    """
    # calculate lengths
    # Your_Code_Here
    netid_lengths = df['netid'].astype(str).str.len()
    ruid_lengths = df['ruid'].astype(str).str.len()

    # find netid with exactly 18 characters and ruid length between 8 and 14
    # Your_Code_Here
    swapped_df = df[(netid_lengths == 18) & ruid_lengths.between(8, 14)].copy()
    
    return swapped_df

In [ ]:
def join_swapped_records(df_swapped, student_list_df):
    """
    Join the swapped records with the student list to correct netid and ruid
    IN: df_swapped, pd.DataFrame, dataframe containing swapped records
    IN: student_list_df, pd.DataFrame, dataframe containing student list
    OUT: joined_df, pd.DataFrame, dataframe with corrected netid and ruid
    """
    # Your_Code_Here
    joined_df = df_swapped.merge(student_list_df,left_on='ruid',right_on='netid',how='inner',suffixes=('_assessment', '_student'))

    return joined_df

### DataFrame Schema